Dataset: CoNLL-2003  
Task: Token Classification (Chunking)

Label Categories:
- B-NP (Beginning of Noun Phrase)
- I-NP (Inside Noun Phrase)
- B-VP (Verb Phrase)
- I-VP
- O (Outside any phrase)

In [3]:
!pip install transformers -q

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [5]:
sentences = [
    ["John", "works", "at", "Google"],
    ["She", "lives", "in", "California"],
    ["Microsoft", "is", "a", "company"]
]

labels = [
    ["B-NP", "B-VP", "O", "B-NP"],
    ["B-NP", "B-VP", "O", "B-NP"],
    ["B-NP", "O", "O", "O"]
]

We tokenize sentences using BERT tokenizer.
We align labels with tokens.
Special tokens and subwords are handled using -100.

In [6]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
label_list = ["B-NP", "B-VP", "O"]

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [8]:
def tokenize_and_align(sentence, label):
    tokenized = tokenizer(sentence, is_split_into_words=True, truncation=True)

    word_ids = tokenized.word_ids()
    aligned_labels = []

    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label2id[label[word_idx]])
        else:
            aligned_labels.append(-100)
        previous_word_idx = word_idx

    tokenized["labels"] = aligned_labels
    return tokenized

dataset = [tokenize_and_align(s, l) for s, l in zip(sentences, labels)]

After preprocessing we get:
- input_ids
- attention_mask
- labels

In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    num_train_epochs=3
)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=0.9615581035614014, metrics={'train_runtime': 11.318, 'train_samples_per_second': 0.795, 'train_steps_per_second': 0.53, 'total_flos': 13780068300.0, 'train_loss': 0.9615581035614014, 'epoch': 3.0})

Model performance is evaluated using Precision, Recall, and F1 Score.
Due to small dataset, values may not be high but demonstrate working pipeline.

In [12]:
sentence = "John works at Google"
inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs)

predictions = outputs.logits.argmax(dim=2)
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, pred in zip(tokens, predictions[0]):
    print(token, id2label[pred.item()])

[CLS] B-VP
john B-NP
works B-NP
at O
google B-NP
[SEP] O


POS Tagging assigns grammatical labels to each word.
Chunking groups words into meaningful phrases.

POS tagging is easier as it works at word level.
Chunking is more complex as it identifies phrases.

Challenges:
- Handling subwords
- Label alignment
- Runtime issues

Observations:
- Transformer models understand context well
- Even small datasets can demonstrate pipeline